In [6]:
import numpy as np
import pandas as pd

from scipy.stats import t
from scipy.optimize import minimize_scalar

pd.set_option("display.float_format", lambda x: f"{x:,.8g}")

EPS = 1e-12
THETA_EPS = 1e-6
RANDOM_SEED = 2026

N_STARTS = 50
MAX_ITER = 1000
TOL = 1e-9

BMW_FILE = "../Data/BMW_DE_std_log_returns.csv"
SIEMENS_FILE = "../Data/SIE_DE_std_log_returns.csv"

# Read the complete files
bmw = (
    pd.read_csv(BMW_FILE, parse_dates=["Date"])
    .rename(columns={"Close": "bmw_return"})
)

siemens = (
    pd.read_csv(SIEMENS_FILE, parse_dates=["Date"])
    .rename(columns={"Close": "siemens_return"})
)

# Use the whole common period; no START_DATE or END_DATE filter
data = (
    bmw.merge(siemens, on="Date", how="inner")
       .dropna(subset=["bmw_return", "siemens_return"])
       .sort_values("Date")
       .reset_index(drop=True)
)

print(f"Number of paired observations: {len(data)}")
print(f"First date: {data['Date'].min().date()}")
print(f"Last date : {data['Date'].max().date()}")

display(data.head())
display(data.tail())

Number of paired observations: 7548
First date: 1996-11-11
Last date : 2026-05-04


,Date,bmw_return,siemens_return
0,1996-11-11,-0.14253704,-0.29827253
1,1996-11-12,0.34347123,0.34891551
2,1996-11-13,-0.089777725,0.14045398
3,1996-11-14,0.13921304,-0.019003712
4,1996-11-15,0.37114517,-0.26711181


,Date,bmw_return,siemens_return
7543,2026-04-27,-0.038224549,1.5314109
7544,2026-04-28,0.021651399,-0.14104135
7545,2026-04-29,-1.1040865,-0.79395992
7546,2026-04-30,0.20569771,0.9996864
7547,2026-05-04,-1.1869075,0.24064325


In [7]:
# ============================================================
# 3. Fit Student-t margins and calculate pseudo-observations
# ============================================================

x = data["bmw_return"].to_numpy(dtype=float)
y = data["siemens_return"].to_numpy(dtype=float)

# The CSV files already contain standardized log returns.
# Do not standardize x and y again here.

df_bmw, loc_bmw, scale_bmw = t.fit(x)
df_sie, loc_sie, scale_sie = t.fit(y)

u = t.cdf(
    x,
    df_bmw,
    loc=loc_bmw,
    scale=scale_bmw
)

v = t.cdf(
    y,
    df_sie,
    loc=loc_sie,
    scale=scale_sie
)

u = np.clip(u, EPS, 1.0 - EPS)
v = np.clip(v, EPS, 1.0 - EPS)

margin_results = pd.DataFrame(
    {
        "distribution": ["Student-t", "Student-t"],
        "df": [df_bmw, df_sie],
        "location": [loc_bmw, loc_sie],
        "scale": [scale_bmw, scale_sie],
    },
    index=["BMW", "Siemens"],
)

print("\nStudent-t marginal estimates")
display(margin_results)

# ============================================================
# 4. Copula log-density functions
# ============================================================

def safe_log_density(values):
    return np.nan_to_num(
        values,
        nan=-1e300,
        neginf=-1e300,
        posinf=1e300,
    )


def logpdf_clayton(u, v, theta):
    """
    Clayton copula log-density.

    theta = 0 is the independence limit.
    """

    if theta <= THETA_EPS:
        return np.zeros_like(u)

    log_u = np.log(u)
    log_v = np.log(v)

    with np.errstate(
        over="ignore",
        divide="ignore",
        invalid="ignore"
    ):
        s = np.exp(-theta * log_u) + np.exp(-theta * log_v) - 1.0

        log_density = (
            np.log1p(theta)
            - (theta + 1.0) * (log_u + log_v)
            - (2.0 + 1.0 / theta) * np.log(s)
        )

    return safe_log_density(log_density)


def logpdf_gumbel(u, v, theta):
    """
    Gumbel copula log-density.

    theta >= 1.
    theta = 1 is the independence limit.
    """

    if theta < 1.0:
        return np.full_like(u, -1e300)

    x1 = -np.log(u)
    x2 = -np.log(v)

    with np.errstate(
        over="ignore",
        divide="ignore",
        invalid="ignore"
    ):
        a = x1**theta + x2**theta
        a_inv = a ** (-1.0 / theta)

        log_density = (
            -a_inv
            + (theta - 1.0) * (np.log(x1) + np.log(x2))
            - np.log(u)
            - np.log(v)
            + (2.0 / theta - 2.0) * np.log(a)
            + np.log1p((theta - 1.0) * a_inv)
        )

    return safe_log_density(log_density)


def logpdf_frank(u, v, theta):
    """
    Numerically stable Frank copula log-density.

    theta > 0.
    theta = 0 is the independence limit.
    """

    if theta <= THETA_EPS:
        return np.zeros_like(u)

    with np.errstate(
        over="ignore",
        divide="ignore",
        invalid="ignore"
    ):
        # Stable calculation of 1 - exp(-theta)
        a = -np.expm1(-theta)

        # Stable calculations of 1 - exp(-theta*u)
        b = -np.expm1(-theta * u)
        c = -np.expm1(-theta * v)

        denominator = a - b * c
        denominator = np.maximum(denominator, EPS)

        log_density = (
            np.log(theta)
            + np.log(a)
            - theta * (u + v)
            - 2.0 * np.log(denominator)
        )

    return safe_log_density(log_density)

# ============================================================
# 6. EM algorithm
# ============================================================

def fit_mixture_em(
    u,
    v,
    first_component="gumbel",
    n_starts=50,
    max_iter=1000,
    tol=1e-9,
    seed=2026
):
    """
    Estimate

        c(u,v)
        =
        w*c_first(u,v; theta1)
        +
        (1-w)*c_clayton(u,v; theta2)
    """

    if first_component == "gumbel":
        first_logpdf = logpdf_gumbel
        first_bounds = (1.0 + THETA_EPS, 30.0)

    elif first_component == "frank":
        first_logpdf = logpdf_frank
        first_bounds = (THETA_EPS, 30.0)

    else:
        raise ValueError(
            "first_component must be 'gumbel' or 'frank'."
        )

    clayton_bounds = (THETA_EPS, 30.0)

    rng = np.random.default_rng(seed)
    best_result = None

    for start in range(n_starts):

        w = float(rng.uniform(0.05, 0.95))
        theta1 = float(rng.uniform(*first_bounds))
        theta2 = float(rng.uniform(*clayton_bounds))

        previous_loglikelihood = -np.inf
        converged = False

        for iteration in range(1, max_iter + 1):

            # ------------------------------------------------
            # E-step
            # ------------------------------------------------

            log_component_1 = (
                np.log(w)
                + first_logpdf(u, v, theta1)
            )

            log_component_2 = (
                np.log1p(-w)
                + logpdf_clayton(u, v, theta2)
            )

            log_mixture_density = np.logaddexp(
                log_component_1,
                log_component_2
            )

            if not np.all(
                np.isfinite(log_mixture_density)
            ):
                break

            loglikelihood = float(
                np.sum(log_mixture_density)
            )

            responsibility_1 = np.exp(
                log_component_1
                - log_mixture_density
            )

            responsibility_1 = np.clip(
                responsibility_1,
                0.0,
                1.0
            )

            # ------------------------------------------------
            # M-step
            # ------------------------------------------------

            w = float(
                np.clip(
                    responsibility_1.mean(),
                    THETA_EPS,
                    1.0 - THETA_EPS
                )
            )

            theta1 = weighted_theta_mle(
                first_logpdf,
                u,
                v,
                responsibility_1,
                first_bounds
            )

            theta2 = weighted_theta_mle(
                logpdf_clayton,
                u,
                v,
                1.0 - responsibility_1,
                clayton_bounds
            )

            # ------------------------------------------------
            # Convergence check
            # ------------------------------------------------

            if np.isfinite(previous_loglikelihood):

                difference = abs(
                    loglikelihood
                    - previous_loglikelihood
                )

                if difference < tol * (
                    1.0
                    + abs(previous_loglikelihood)
                ):
                    converged = True
                    break

            previous_loglikelihood = loglikelihood

        # ----------------------------------------------------
        # Final log-likelihood
        # ----------------------------------------------------

        final_log_component_1 = (
            np.log(w)
            + first_logpdf(u, v, theta1)
        )

        final_log_component_2 = (
            np.log1p(-w)
            + logpdf_clayton(u, v, theta2)
        )

        final_loglikelihood = float(
            np.sum(
                np.logaddexp(
                    final_log_component_1,
                    final_log_component_2
                )
            )
        )

        # Reject invalid numerical solutions
        if not np.isfinite(final_loglikelihood):
            continue

        if abs(final_loglikelihood) > 1e8:
            continue

        result = {
            "model": (
                f"{first_component.title()}-Clayton"
            ),
            "w": w,
            "theta1": theta1,
            "theta2": theta2,
            "log_likelihood": final_loglikelihood,
            "iterations": iteration,
            "start": start + 1,
            "converged": converged
        }

        if (
            best_result is None
            or final_loglikelihood
            > best_result["log_likelihood"]
        ):
            best_result = result

    if best_result is None:
        raise RuntimeError(
            f"No valid solution found for "
            f"{first_component.title()}-Clayton."
        )

    return best_result

# ============================================================
# 7. Estimate both mixture copulas
# ============================================================

gumbel_clayton_result = fit_mixture_em(
    u,
    v,
    first_component="gumbel",
    n_starts=N_STARTS,
    max_iter=MAX_ITER,
    tol=TOL,
    seed=RANDOM_SEED
)

frank_clayton_result = fit_mixture_em(
    u,
    v,
    first_component="frank",
    n_starts=N_STARTS,
    max_iter=MAX_ITER,
    tol=TOL,
    seed=RANDOM_SEED + 1
)


# ============================================================
# 8. Results table
# ============================================================

results = pd.DataFrame(
    [
        gumbel_clayton_result,
        frank_clayton_result
    ]
)

number_of_parameters = 3
n = len(data)

results["AIC"] = (
    -2.0 * results["log_likelihood"]
    + 2.0 * number_of_parameters
)

results["BIC"] = (
    -2.0 * results["log_likelihood"]
    + number_of_parameters * np.log(n)
)

results = results.sort_values(
    "log_likelihood",
    ascending=False
).reset_index(drop=True)

print("\nMixture-copula estimation results")
display(results)


# ============================================================
# 9. Save results
# ============================================================

results.to_csv(
    "BMW_Siemens_mixture_copula_results_full_period.csv",
    index=False
)

print(
    "Saved: "
    "BMW_Siemens_mixture_copula_results_full_period.csv"
)


Student-t marginal estimates


,distribution,df,location,scale
BMW,Student-t,3.218432,0.001837245,0.66213417
Siemens,Student-t,2.9177101,0.0047255154,0.57053391



Mixture-copula estimation results


,model,w,theta1,theta2,log_likelihood,iterations,start,converged,AIC,BIC
0,Gumbel-Clayton,0.72172605,1.000001,7.8155445,"9,664.71",27,46,True,"-19,323.42","-19,302.633"
1,Frank-Clayton,0.41515443,11.431198,0.42376526,"1,540.5239",204,1,True,"-3,075.0478","-3,054.2607"


Saved: BMW_Siemens_mixture_copula_results_full_period.csv
